# Export Final TIFFs

This notebook selects processed images from one or more LIF folders, exports each matching LIF scene once, assigns sequential names, and creates an `export_manifest.csv` table relating the original and new names.

A scene is included when its analysis directory contains at least one `tracking_*.csv` file. Multiple suffixed result directories for the same scene—for example, separate cell analyses—are grouped together and still produce only one TIFF.

## 1. Start Jupyter with the MicroLive kernel

Open a terminal and run:

```bash
cd /Users/nzlab-la/Desktop/microlive
/opt/anaconda3/bin/jupyter lab
```

Then open `notebooks/export_data/Export_Final_tifs.ipynb`. It is configured to use the installed `Python (microlive)` kernel. If Jupyter asks you to choose a kernel, select **Python (microlive)**, then run the cells from top to bottom.

If the MicroLive kernel is not listed, close Jupyter and register it once:

```bash
conda activate microlive
python -m ipykernel install --user --name microlive --display-name "Python (microlive)"
conda deactivate
```

Expected input organization:

```text
my_dataset/
├── LIFs/
│   ├── experiment_01.lif
│   └── experiment_02.lif
└── HT_Analysis_GUI/
    ├── results_experiment_01_Image1_cellA/
    │   └── tracking_....csv
    └── results_experiment_01_Image1_cellB/
        └── tracking_....csv
```

The LIF files must currently be directly inside each configured `lif_folder`; nested LIF subfolders are not searched.

In [ ]:
# Load the existing export code. No data are read or written in this cell.
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

if "microlive" not in Path(sys.prefix).name.lower():
    raise RuntimeError(
        "This notebook must use the 'microlive' conda environment. "
        "Close Jupyter, run 'conda activate microlive', and start Jupyter again."
    )

# Locate run_filter_and_rename.py whether Jupyter started in the repository
# root or in this notebook's directory.
candidate_script_folders = [
    Path.cwd(),
    Path.cwd() / "notebooks" / "export_data",
    Path("/Users/nzlab-la/Desktop/microlive/notebooks/export_data"),
]
SCRIPT_FOLDER = next(
    (folder.resolve() for folder in candidate_script_folders
     if (folder / "run_filter_and_rename.py").is_file()),
    None,
)
if SCRIPT_FOLDER is None:
    raise FileNotFoundError(
        "Could not find run_filter_and_rename.py. Start Jupyter from the "
        "MicroLive repository or add its folder to candidate_script_folders."
    )

if str(SCRIPT_FOLDER) not in sys.path:
    sys.path.insert(0, str(SCRIPT_FOLDER))

import run_filter_and_rename as exporter

print(f"Python environment: {sys.prefix}")
print(f"Exporter loaded from: {SCRIPT_FOLDER / 'run_filter_and_rename.py'}")

## 2. Configure the export

Edit the paths and options in the next cell. `DATA_ROOT_DIR`, `lif_folder`, `results_folder`, and `OUTPUT_BASE_DIR` must be real folders on your computer.

Leave `DRY_RUN = True` for the first run. A dry run reads LIF metadata and displays what would be kept, skipped, and renamed, but it does not create TIFFs, AVIs, or manifests.

In [ ]:
# ============================================================
# ALL EXPORT ARGUMENTS — edit this cell for your experiment
# ============================================================

# Main dataset location. Replace this with your actual mounted drive/folder.
DATA_ROOT_DIR = Path("/Volumes/LaCie/test_export/UTag")

# One entry is needed for each LIF/results folder pair.
# Add more entries using the same structure when processing multiple datasets.
DATASETS = {
    "UTag": {
        # Folder containing .lif files directly (not recursively).
        "lif_folder": DATA_ROOT_DIR / "LIFs",
        # Folder containing results_* analysis directories.
        "results_folder": DATA_ROOT_DIR / "HT_Analysis_GUI",
        # Prefix used for sequential output names: UTag_HT_001, etc.
        "rename_prefix": "UTag_HT",
    },
    # Example second dataset (remove # characters and edit paths to use it):
    # "AlfaTag": {
    #     "lif_folder": Path("/path/to/AlfaTag/LIFs"),
    #     "results_folder": Path("/path/to/AlfaTag/HT_Analysis_GUI"),
    #     "rename_prefix": "AlfaTag_HT",
    # },
}

# Root output folder. Each DATASETS entry gets its own subfolder here.
OUTPUT_BASE_DIR = DATA_ROOT_DIR / "exported_filtered"

# Sequential naming settings. Numbering restarts for each dataset entry.
START_INDEX = 1       # First output number.
ZERO_PAD = 3          # 3 -> 001; 4 -> 0001.

# File types to create.
EXPORT_TIF = True     # Export OME-TIFF files.
EXPORT_VIDEO = False  # Export AVI files in addition to TIFFs.

# TIFF Z behavior.
MAX_PROJECTION = True # True: collapse Z with max projection; False: preserve Z.

# AVI options. AVIs always use a max-Z projection of one selected channel.
VIDEO_FPS = 10
VIDEO_CHANNEL = 0          # Zero-based channel: 0 is the first channel.
VIDEO_SHOW_TIMESTAMP = False
VIDEO_SHOW_SCALEBAR = False
VIDEO_MIN_PERCENTILE = 0.1 # Display contrast lower percentile.
VIDEO_MAX_PERCENTILE = 99.9 # Display contrast upper percentile.
VIDEO_SIGMA = 0.7          # Gaussian smoothing; use 0 to disable.
VIDEO_LOW_SIGMA = 0.15     # Low Gaussian smoothing; use 0 to disable.
VIDEO_DPI = 150

# Safety controls.
DRY_RUN = True             # Always preview first; set False only after review.
ALLOW_EXISTING_OUTPUT = False  # False prevents accidental overwrite of prior output.

# Current limitation: TIFF and AVI exports include every time frame in the
# selected LIF scene. Start/end frame cropping is not yet implemented.

print(f"Configured {len(DATASETS)} dataset(s). DRY_RUN={DRY_RUN}")

## 3. Validate the paths

Run the next cell before every export. It stops on missing folders, reports how many LIF files and processed result directories were found, and protects existing output unless you explicitly allow it.

In [ ]:
validation_rows = []
validation_errors = []

for dataset_name, config in DATASETS.items():
    lif_dir = Path(config["lif_folder"]).expanduser().resolve()
    results_dir = Path(config["results_folder"]).expanduser().resolve()
    output_dir = Path(OUTPUT_BASE_DIR).expanduser().resolve() / dataset_name

    if not lif_dir.is_dir():
        validation_errors.append(f"{dataset_name}: missing LIF folder: {lif_dir}")
        lif_file_count = 0
    else:
        lif_file_count = len([
            path for path in lif_dir.glob("*.lif")
            if not path.name.startswith("._")
        ])
        if lif_file_count == 0:
            validation_errors.append(f"{dataset_name}: no .lif files in {lif_dir}")

    if not results_dir.is_dir():
        validation_errors.append(
            f"{dataset_name}: missing results folder: {results_dir}"
        )
        processed_result_count = 0
    else:
        processed_result_count = len(exporter._build_processed_stems(results_dir))
        if processed_result_count == 0:
            validation_errors.append(
                f"{dataset_name}: no results_* folders containing tracking_*.csv"
            )

    has_existing_output = output_dir.exists() and any(output_dir.iterdir())
    if not DRY_RUN and has_existing_output and not ALLOW_EXISTING_OUTPUT:
        validation_errors.append(
            f"{dataset_name}: output is not empty: {output_dir}. "
            "Choose a new OUTPUT_BASE_DIR or set ALLOW_EXISTING_OUTPUT=True."
        )

    validation_rows.append({
        "Dataset": dataset_name,
        "LIF_Folder": str(lif_dir),
        "LIF_Files": lif_file_count,
        "Results_Folder": str(results_dir),
        "Processed_Result_Folders": processed_result_count,
        "Output_Folder": str(output_dir),
        "Output_Already_Exists": has_existing_output,
    })

display(pd.DataFrame(validation_rows))

if validation_errors:
    raise RuntimeError("Fix these configuration problems:\n- " + "\n- ".join(validation_errors))

print("Validation passed. You can run the export cell.")

## 4. Preview or run the export

With `DRY_RUN = True`, the cell prints the proposed mapping without writing files. Check every `KEEP`, `SKIP`, and unmatched-folder warning.

When the preview is correct, return to the configuration cell, set `DRY_RUN = False`, run the validation cell again, and then rerun this cell.

In [ ]:
video_options = {
    "show_timestamp": VIDEO_SHOW_TIMESTAMP,
    "show_scalebar": VIDEO_SHOW_SCALEBAR,
    "fps": VIDEO_FPS,
    "channel": VIDEO_CHANNEL,
    "min_percentile": VIDEO_MIN_PERCENTILE,
    "max_percentile": VIDEO_MAX_PERCENTILE,
    "sigma": VIDEO_SIGMA,
    "low_sigma": VIDEO_LOW_SIGMA,
    "dpi": VIDEO_DPI,
}

manifest_dataframes = []
run_summary = []

for dataset_name, config in DATASETS.items():
    normalized_config = {
        "lif_folder": str(Path(config["lif_folder"]).expanduser().resolve()),
        "results_folder": str(Path(config["results_folder"]).expanduser().resolve()),
        "rename_prefix": config["rename_prefix"],
    }

    kept_count, skipped_count, manifest_df = exporter._process_tag(
        tag_name=dataset_name,
        config=normalized_config,
        output_base_dir=Path(OUTPUT_BASE_DIR).expanduser().resolve(),
        start_index=START_INDEX,
        zero_pad=ZERO_PAD,
        should_export_tif=EXPORT_TIF,
        should_export_video=EXPORT_VIDEO,
        should_max_project=MAX_PROJECTION,
        is_dry_run=DRY_RUN,
        video_options=video_options,
    )

    run_summary.append({
        "Dataset": dataset_name,
        "Kept_Scenes": kept_count,
        "Skipped_Scenes": skipped_count,
        "Mode": "DRY RUN" if DRY_RUN else "EXPORTED",
    })
    if manifest_df is not None and not manifest_df.empty:
        manifest_df = manifest_df.copy()
        manifest_df.insert(0, "Dataset", dataset_name)
        manifest_dataframes.append(manifest_df)

print("\nRun summary")
display(pd.DataFrame(run_summary))

if manifest_dataframes:
    combined_manifest_df = pd.concat(manifest_dataframes, ignore_index=True)
    print("Proposed/exported name mapping")
    display(combined_manifest_df)
else:
    combined_manifest_df = pd.DataFrame()
    print("No matching scenes were found.")

if DRY_RUN:
    print("DRY RUN complete: no files were written.")
    print("Set DRY_RUN=False in the configuration cell after reviewing the table.")
else:
    print(f"Export complete. Output root: {Path(OUTPUT_BASE_DIR).expanduser().resolve()}")

## Output layout

For the `UTag` example, completed output is organized as:

```text
exported_filtered/
└── UTag/
    ├── tif/
    │   ├── UTag_HT_001_maxZ.ome.tif
    │   └── UTag_HT_002_maxZ.ome.tif
    ├── avi/
    │   ├── UTag_HT_001.avi
    │   └── UTag_HT_002.avi
    └── export_manifest.csv
```

`Matching_Results_Count` and `Matching_Results_Folders` in the manifest show when multiple cell-specific result directories were consolidated into one TIFF.

Important: the exporter writes complete time series. It does not currently stop at a chosen frame. `MAX_PROJECTION` controls TIFF Z projection; AVI output always uses a max-Z projection of `VIDEO_CHANNEL`.